# Trash-bin-detection — Colab-arbeidsflyt

**Branch:** `streetview-yolo-retry`  
**Data:** hentes fra Google Drive (`data.zip`)  

### Seksjoner
1. ⚙️ Oppsett (kjør alltid disse)
2. 🌍 Hent Street View-bilder
3. 🏷️ Annotere (krever lokal Colab-kjøring for GUI)
4. 🔍 Segmenteringsgjennomgang (valgfri)
5. 🚀 Tren YOLO-modell
6. 📊 Evaluer modell
7. 💾 Lagre resultater til Drive

---
## ⚙️ 1. Oppsett
Kjør disse cellene én gang per Colab-sesjon.

In [ ]:
# ── Variabler du kan endre ─────────────────────────────────────────────────
REPO_URL    = "https://github.com/davgei/trash-bin-detection.git"
BRANCH      = "streetview-yolo-retry"
REPO_DIR    = "/content/trash-bin-detection"

# Sti til data.zip på Google Drive (relativ til Drive-rota)
DRIVE_ZIP   = "/content/drive/MyDrive/data.zip"

# Google Maps API-nøkkel (brukes bare av fetch-skriptene)
# Legg den inn her ELLER bruk Colab Secrets (anbefalt):
#   Venstre sidemeny → Nøkkel-ikon → legg til GOOGLE_MAPS_API_KEY
GOOGLE_MAPS_API_KEY = ""   # ← lim inn nøkkel her, eller la stå tom
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

# Klon repo (eller oppdater hvis den allerede finnes)
if os.path.isdir(REPO_DIR):
    print("Repo finnes — henter siste endringer ...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
        check=True
    )

%cd {REPO_DIR}
print("Arbeidsmappe:", os.getcwd())

In [ ]:
import zipfile, shutil

# Pakk ut data.zip fra Drive til repo-rota
# Eksisterende data/-mappe erstattes IKKE — nye filer legges til.
print(f"Pakker ut {DRIVE_ZIP} → {REPO_DIR}/")
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(REPO_DIR)
print("Ferdig.")

# Sjekk at vi fikk det vi forventet
for d in ["data/annotated_backup", "data/annotated", "data/to_annotate"]:
    exists = os.path.isdir(d)
    print(f"  {'✓' if exists else '✗'}  {d}")

In [ ]:
# Installer avhengigheter
!pip install -q ultralytics opencv-python-headless requests
# transformers trengs for SAM2-seg-seksjonen (stor nedlasting)
# !pip install -q transformers

# Sjekk GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "Ingen GPU funnet"

In [ ]:
# Sett API-nøkkel (hent fra Colab Secrets hvis ikke satt over)
if not GOOGLE_MAPS_API_KEY:
    try:
        from google.colab import userdata
        GOOGLE_MAPS_API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
        print("API-nøkkel lest fra Colab Secrets.")
    except Exception:
        print("⚠️  Ingen API-nøkkel funnet. Fetch-cellene vil feile.")

os.environ['GOOGLE_MAPS_API_KEY'] = GOOGLE_MAPS_API_KEY or ""

---
## 🌍 2. Hent Street View-bilder

Last opp CSV-filen til Drive og juster stien under.  
Bilder havner i `data/to_annotate/` og logges til `data/streetview_log.csv`.

In [ ]:
# ── Innstillinger ──────────────────────────────────────────────────────────
# CSV eksportert fra hentesteder-systemet (Uttrekk_products eller hentesteder)
CSV_PATH   = "data/Uttrekk_products(result).csv"  # ← endre om nødvendig

# Valgfri: bruk en trent modell for å sjekke om beholderen er synlig
YOLO_MODEL = "models/trained/trash_bin_yolo11n_best.pt"  # sett til "" for å hoppe over

# Maks antall bilder å hente (None = alle)
MAX_FETCH  = 50
# ──────────────────────────────────────────────────────────────────────────

!python -m src.fetch_streetview_from_csv \
    --csv {CSV_PATH} \
    --limit {MAX_FETCH} \
    --model {YOLO_MODEL}

In [ ]:
# Vis statistikk over siste fetch-runde
!python -m src.streetview_stats

In [ ]:
# Tell bilder som venter på annotering
import glob
images = glob.glob("data/to_annotate/*.jpg") + glob.glob("data/to_annotate/*.png")
labels = glob.glob("data/to_annotate/*.txt")
print(f"Bilder i to_annotate/: {len(images)}")
print(f"Allerede annotert:     {len(labels)}")
print(f"Gjenstår:              {len(images) - len(labels)}")

---
## 🏷️ 3. Annotere

Annotasjonsverktøyet (`annotate.py`) bruker et interaktivt OpenCV-vindu med
tastatursnarveier. Det krever en skjerm.

### Alternativ A — Anbefalt: lokal Colab-kjøring
1. Installer Jupyter lokalt: `pip install notebook`
2. Kjør: `jupyter notebook --no-browser --port=8888`
3. I Colab: klikk pil ned ved **Koble til** → **Koble til en lokal kjøring**  
   → lim inn `http://localhost:8888/?token=...`
4. Kjør cellen under direkte — vinduer åpnes på din maskin.

### Alternativ B — Sky-Colab med virtuell skjerm (begrenset)
Med sky-runtime kan du kjøre skriptet, men du ser ikke vinduet i nettleseren.
Eneste praktiske bruk: batch-eksport uten GUI (se lengre ned).

---
**Tastesnarveier i assistert modus:**

| Tast | Handling |
|------|----------|
| `a` / Enter | Godta YOLO-forslag |
| `r` | Forkast og tegn manuelt |
| `b` | Merk som bakgrunn (ingen beholdere) |
| `s` | Hopp over dette bildet |
| `q` | Avslutt og eksporter |

Alle etiketter eksporteres automatisk til `data/annotated_backup/` når du avslutter.

In [ ]:
import sys, subprocess as _sp

# Sjekk om vi kjører i sky-Colab (ikke lokal runtime)
_in_colab = "google.colab" in sys.modules

if _in_colab:
    _sp.run(["apt-get", "install", "-qq", "xvfb"], check=True)
    _sp.run(["pkill", "Xvfb"], capture_output=True)   # rydd opp evt. gammel prosess
    _sp.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24"])
    os.environ["DISPLAY"] = ":99"
    print("Virtuell skjerm startet på :99")
    print("⚠️  Du ser ikke annotasjonsvinduet — bruk lokal kjøring for interaktiv annotering.")
else:
    print("Lokal kjøring detektert — GUI fungerer normalt.")

In [ ]:
# ── Kjør annotasjonsverktøyet (assistert modus) ────────────────────────────
ANNOT_MODEL = "models/trained/trash_bin_yolo11n_best.pt"
CONF        = 0.25   # YOLO-konfidansterskelen for forslag

!python -m src.labeling.annotate \
    --mode assisted \
    --model {ANNOT_MODEL} \
    --conf {CONF}

In [ ]:
# ── Rebuild datasett-splitt etter annotering ───────────────────────────────
# (kjøres automatisk av annotate.py, men kan kjøres manuelt om nødvendig)
!python -m src.prepare_dataset

---
## 🔍 4. Segmenteringsgjennomgang (valgfri)

Bruker SAM2 + ADE20K for å auto-generere segmenteringsmasker, etterfulgt av
interaktiv gjennomgang. Krever `transformers`-biblioteket (installert i cellen under).

Samme GUI-begrensning som i seksjon 3 gjelder.

In [ ]:
!pip install -q transformers

In [ ]:
# Auto-generer segmenteringsetiketter for alle bilder i annotated/
# Resultater lagres i data/annotated_seg/
!python -m src.labeling.sam2_seg_autolabel

In [ ]:
# Interaktiv gjennomgang av segmenteringsetiketter
# Tastesnarveier: a/Enter = godta, s = hopp over, f = flagg, q = avslutt
!python -m src.labeling.sam2_seg_review

---
## 🚀 5. Tren YOLO-modell

Velg runtime med GPU (**Kjøretid → Endre kjøretidstype → T4 GPU**) før du kjører denne.

In [ ]:
# ── Treningsinnstillinger ──────────────────────────────────────────────────
BASE_MODEL = "yolo11n.pt"   # utgangspunkt (lastes ned automatisk om ikke tilstede)
EPOCHS     = 100
IMG_SIZE   = 640
BATCH      = 16
RUN_NAME   = "colab_run"
PATIENCE   = 20
# ──────────────────────────────────────────────────────────────────────────

!python -m src.train \
    --model {BASE_MODEL} \
    --epochs {EPOCHS} \
    --imgsz {IMG_SIZE} \
    --batch {BATCH} \
    --name {RUN_NAME} \
    --patience {PATIENCE}

In [ ]:
# Finn de nye beste vektene
import glob as _glob
weights = sorted(_glob.glob(f"runs/detect/models/trained/{RUN_NAME}/weights/best.pt"))
if weights:
    TRAINED_MODEL = weights[-1]
    print("Beste vekter:", TRAINED_MODEL)
else:
    print("⚠️  Ingen vekter funnet — sjekk at treningen fullførte.")
    TRAINED_MODEL = ""

---
## 📊 6. Evaluer modell

In [ ]:
# Evaluer på testsplitt (aldri sett under trening)
EVAL_MODEL = TRAINED_MODEL or "models/trained/trash_bin_yolo11n_best.pt"

!python -m src.evaluate --model {EVAL_MODEL}

---
## 💾 7. Lagre resultater til Google Drive

Synkroniserer modellvekter og oppdatert `data/` tilbake til Drive.

In [ ]:
import shutil, pathlib

DRIVE_DEST = "/content/drive/MyDrive/trash-bin-detection-backup"
os.makedirs(DRIVE_DEST, exist_ok=True)

# Kopier trente modeller
if TRAINED_MODEL and os.path.isfile(TRAINED_MODEL):
    dest = os.path.join(DRIVE_DEST, f"{RUN_NAME}_best.pt")
    shutil.copy2(TRAINED_MODEL, dest)
    print("Vekter lagret til Drive:", dest)

print("Ferdig.")

In [ ]:
# Pakk og last opp oppdatert data.zip til Drive
# (inkluderer annotated_backup, annotated, streetview_log, to_annotate)

print("Pakker data/ → data_updated.zip ...")
shutil.make_archive("/content/data_updated", 'zip', REPO_DIR, 'data')

dest_zip = os.path.join(DRIVE_DEST, "data_updated.zip")
shutil.move("/content/data_updated.zip", dest_zip)
print("Lastet opp til:", dest_zip)